## Settings

In [47]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [48]:
## libraries
import sys
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import load_processed_data
from src.estimators.factories import load_estimators
from src.evaluators.resampling import logo_cross_valid
from src.evaluators.recovering import (
    compile_structural_recovery,
    results_structural_recovery,
)

## constants
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z,
    TARGET,
)

## Initialization

In [49]:
## reproducibility
N_REPEATS = 30
RANDOM_STATE = 42

## load data and models
data = load_processed_data()
models = load_estimators(random_state = RANDOM_STATE)

## view data shape
n_obs, n_feat = data.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")

## view model surface
n_mods = len(models)
print(f"Learned Models: {n_mods} estimators")

Original Data: 32 features, 25 observations
Learned Models: 9 estimators


## Training

In [50]:
## leave-one-group-out cross validation (domain)
if "results_dict_domain" not in globals():
    results_dict_domain = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        _, y_pred = logo_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            group = "domain",
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        results_dict_domain[name] = y_pred

## Post-Processing

In [51]:
## consensus metrics per (model, held-out domain)
results_data_domain = compile_structural_recovery(
    predictions = results_dict_domain,
    data = data,
    target = TARGET,
    group = "domain"
)

## Structural Recovery Tests
Leave-one-group-out (LOGO) cross-validation by domain evaluates whether held-out predictions recover the structural ordering and dependence of observed capacities. Because frontier estimation is scale-arbitrary across fitting methods, recovery is assessed via structural agreement rather than residual norms.

- **Spearman (`ρ`)**: Global rank agreement between predicted and observed capacities.
- **Rank-Biased Overlap (`RBO`)**: Top-weighted agreement, emphasizing the high-capacity tail.
- **Distance Correlation (`DCR`)**: Nonlinear dependence between predictions and observations.
- **Composite Index (`CI`)**: Geometric mean of agreement-scaled `ρ`, `RBO`, and `DCR`.

### Reporting Convention
Each held-out domain contributes one value per estimator. The summary table reports held-out domains row-wise. `CI` is shown as median `[Q1–Q3]` across estimators; component columns report medians. Prior to compounding, `ρ` is rescaled from `[-1, 1]` to `[0, 1]` so that negative rank recovery lowers `CI` without collapsing to a single floor. Aggregations are equally weighted across estimators. Higher values indicate stronger recovery.

In [52]:
## structural recovery results by held-out domain
results_domain = results_structural_recovery(
    results = results_data_domain,
    group_col = "group",
    index_name = "Domain",
    n_repeats = N_REPEATS,
    random_state = RANDOM_STATE,
    decimals = 2,
)

display(results_domain)

Cross-Validation: 9 models, 30 repeats (seeds 42-71)
Across-model aggregation: median across learners within held-out domains
Resampling: LOGO domain splits are fixed across repeats
Weighting: domains and models are equally weighted; results are not observation-weighted


,CI [IQR],ρ,RBO,DCR
Domain,,,,
Earth & Physical Sciences,"0.88 [0.81, 0.91]",0.70,0.90,0.84
Life Sciences & Medicine,"0.92 [0.65, 0.94]",0.80,0.88,0.83
Technology & Information,"0.87 [0.82, 0.94]",0.80,0.93,0.89
Trade & Institutions,"0.96 [0.81, 0.97]",0.90,0.98,0.94
Transportation & Infrastructure,"0.83 [0.68, 0.95]",0.60,0.85,0.83


_Structural recovery is consistent across held-out domains. Median `CI` ranges from 0.83 to 0.96, with positive rank recovery (median `ρ` between 0.60 and 0.90) and elevated top-end agreement (`RBO`) across all domains. Recovery strength varies by domain but remains positive throughout, indicating that fitted frontiers reproduce the observed capacity structure on unseen domains._

## Visualization